# Earthquake or explosion — how does the world verify a nuclear test ban?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F10_earthquake_or_explosion.ipynb).

The Comprehensive Nuclear-Test-Ban Treaty, opened for signature in 1996, bans every nuclear
explosion anywhere on Earth, and the way a ban like that is checked is that the planet listens.
Seismometers do not care what shook the ground. They record an earthquake, a landslide, a mine
collapse and a bomb in the same way, and somebody has to look at each recording and decide which
of those it was.

California will not hand you a nuclear test. It hands you the same decision, thousands of times a
year, at a smaller scale: quarries blast rock, the seismic network records the blasts alongside
real earthquakes, and a USGS analyst labels every event either `earthquake` or `quarry blast`.
That is tens of thousands of decisions somebody has already made — which is exactly what you need
if you want to find out whether a machine could have made them instead.

Today you build that classifier. Then you find out what it actually learned.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say what an event catalogue does and does not tell you about the source of a
seismic signal, name the things that make a quarry blast recognisable in one, and say which of
them would still be there if the event you were hunting were a secret nuclear test.

**The skills.** Split labelled data into a training half and a held-out half with `stratify`, fit
`LogisticRegression` and `GaussianNB` to it, and judge a classifier with a `confusion_matrix` and
with `precision_score`, `recall_score` and `f1_score` rather than with accuracy.

**Eight places where you write something: five in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it.

**The four questions, in order:**

1. When one event in 47 is a blast, what counts as getting it right?
2. What does a catalogue row actually know about a quarry blast?
3. Can a fitted model beat two conditions you wrote by hand?
4. Would the same numbers survive a different cut of the data?

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

COLUMNS = ['time', 'latitude', 'longitude', 'depth', 'mag', 'type']

# Both files came out of ONE query to the USGS catalogue, asked twice — once for each label:
#
#   https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&orderby=time-asc
#     &starttime=2015-01-01&endtime=2026-08-31&minmagnitude=1.5
#     &minlatitude=32&maxlatitude=42&minlongitude=-125&maxlongitude=-114
#     &eventtype=earthquake   ... and again with &eventtype=quarry%20blast
#
# Unlike most weeks, this one does not run that query for you. It cannot: the archive refuses any
# request matching more than 20,000 events and the earthquake half matches far more, so that half
# has to travel with the course as a file. Fetching only the blast half live would leave you
# joining this week's blasts to last summer's earthquakes — not the same slice of the catalogue,
# and every count in the text below would drift away from what your screen says. So both halves
# ship together, and the numbers you print are the numbers you read.
#
# The archive sends more columns than COLUMNS keeps. Hold on to the whole blast table too: two
# of the columns we are about to drop turn out to say where a blast was and how its depth was
# arrived at, and we come back for them.
blasts_all = pd.read_csv(CACHE + "/week10_ca_quarry_blasts_2015_2026.csv")
blasts = blasts_all[COLUMNS]
quakes = pd.read_csv(CACHE + "/week10_ca_earthquakes_2015_2026.csv.gz")
coast = pd.read_csv(CACHE + "/coastlines.csv")

print("columns:", list(quakes.columns))
print(blasts.head(2))

## When one event in 47 is a blast, what counts as getting it right?

Both files came out of the same query: the same box of California and western Nevada, the same
years, the same magnitude floor. The only difference is what the analyst wrote in the `type`
column.

Stack them into one table. `pd.concat` takes a list of tables that share their columns and
returns one longer table, and `ignore_index=True` renumbers the rows from 0 instead of restarting
the count at the join. While you are there, turn the label into the column a classifier can
actually use: `True` when the event is a blast, `False` when it is an earthquake.

In [ ]:
events = pd.concat([quakes, blasts], ignore_index=True)
events["is_blast"] = events["type"] == "quarry blast"

print(events.shape)
print(events.head(3))

Remember what a file like this is: *A catalogue lists what somebody's instruments recorded, not
what happened. Where there are no seismometers there are no earthquakes in the file.* Every label
in the `type` column was put there by a person, and that is going to matter more than it sounds
like it should.

First, how many of each.

### ✏️ Your turn 1

`events["type"].value_counts()` counts how many rows carry each label. Print it, then print how
many earthquakes this catalogue holds **for every one quarry blast** — one count divided by the
other, rounded to one decimal place.

**Use these names**, because the self-check looks for them: `n_blasts` and `n_quakes`.

In [ ]:
# ← your answer here


assert n_blasts < n_quakes, "blasts are the rare label here — if your n_blasts is the bigger of the two numbers, the two names are the wrong way round"
print("✓ the two classes —", n_blasts, "quarry blasts and", n_quakes,
      "earthquakes, a ratio of", round(n_quakes / n_blasts, 1), "to 1")

That ratio is the whole problem in one number. Quarry blasts are about 46 times
rarer than earthquakes here, and a class that rare changes what you are allowed to call success.

The obvious score is **accuracy**: out of every event, what fraction did you label correctly?
Before you compute it, commit to a number.

### Predict before you run

Here is a classifier that took no effort at all. It ignores its input and answers "earthquake"
every single time, so it never once flags a blast. What accuracy does it get on this catalogue?
Change `my_guess` to a fraction between 0 and 1, then run the cell.

In [ ]:
my_guess = None    # ← your number, written down before you look

In [ ]:
assert my_guess is not None, \
    "write a number into my_guess in the cell above — the commitment is the point, "\
    "and a guess you made before you saw the answer is the only one that can teach you anything"
print("✓ committed — I think", my_guess, "is the accuracy of a rule that always says earthquake")

In [ ]:
always_earthquake = [False] * len(events)
print("you guessed:  ", my_guess)
print("this rule got:", round(accuracy_score(events["is_blast"], always_earthquake), 4))

A rule that is wrong about every single thing you care about is right
97.87% of the time, because 97.87% of the catalogue is the
answer it always gives. That is what a rare class does to accuracy, and it is why nobody who
works on rare events reports accuracy on its own.

The two numbers reported instead split the question in half. *Of the ones you flagged, how many
were right? Of the real ones, how many did you catch?* The first is **precision**, the second is
**recall**, and they pull against each other: flag everything and your recall is perfect while
your precision collapses; flag almost nothing and the reverse. **F1** is the single number that
refuses to let you cheat either way. It is the harmonic mean of precision and recall, which is a
way of saying that it sits near the smaller of the two, so it is only good when both are.

### ✏️ Your turn 2

Score **both** ways of cheating, because the paragraph above claims they fail in opposite
directions and a claim like that is worth checking.

The first is the rule you just ran: `always_earthquake`, which never says blast. The second is its
mirror image — a rule that calls every single event a blast, which is `[True] * len(events)`.

With `events["is_blast"]` as the truth, print the precision, the recall and the F1 of each, using
`precision_score`, `recall_score` and `f1_score`. Six numbers, three per rule.

Each of those three takes `zero_division=0` as a third argument. It tells scikit-learn what to do
when a rule flags nothing at all: there is no *"of the ones you flagged"* left to divide by, so
count it as zero rather than stopping with an error.

**Use these names**, because the self-check looks for them: `always_precision`, `always_recall`
and `always_f1` for the first rule, then `always_blast` for the second rule's predictions and
`blast_precision`, `blast_recall` and `blast_f1` for its three scores.

In [ ]:
# ← your answer here


assert always_recall == 0 and blast_recall == 1, "recall answers 'of the real blasts, how many did you catch?' — a rule that never says blast catches none of them and a rule that always says blast catches every one, so those two recalls have to come out 0 and 1. If yours did not, check which score you put in which name"
print("✓ the two ways to cheat — never saying blast scores accuracy",
      round(accuracy_score(events["is_blast"], always_earthquake), 4), "and F1", always_f1,
      "; always saying blast scores recall", blast_recall, "and F1", round(blast_f1, 4))

## What does a catalogue row actually know about a quarry blast?

Six columns arrived with each event: when it happened, where, how deep, how big, and the label.
Anything a classifier learns has to come out of the first four, so look at them — starting with
*when*, because the times need a moment's work first.

`pd.to_datetime` turns the text into real timestamps. The catalogue records them in UTC, which is
no use for a question about a working day, so `.dt.tz_convert("US/Pacific")` moves them onto the
clock the quarry crew actually works to, and `.dt.hour` and `.dt.dayofweek` read the hour and the
day off each one (Monday is 0, Sunday is 6). Then *where*.

In [ ]:
local = pd.to_datetime(events["time"]).dt.tz_convert("US/Pacific")
events["hour"] = local.dt.hour
events["weekday"] = local.dt.dayofweek

blast_rows = events[events["is_blast"]]
quake_rows = events[events["type"] == "earthquake"]
few_quakes = quake_rows.iloc[::40]      # one earthquake in forty, or every plot is solid ink

print(events[["time", "hour", "weekday", "is_blast"]].head(3))

In [ ]:
plt.figure(figsize=(5, 5))            # California is nearly as tall as it is wide
plt.scatter(few_quakes["longitude"], few_quakes["latitude"], s=2, color="0.7",
            label="earthquakes")
plt.scatter(blast_rows["longitude"], blast_rows["latitude"], s=2, color="firebrick",
            label="blasts")
plt.plot(coast["lon"], coast["lat"], color="0.3", lw=0.6)
plt.xlim(-125, -114)
plt.ylim(32, 42)
plt.gca().set_aspect("equal")
plt.xlabel("degrees east")
plt.ylabel("degrees north")
plt.title(f"{len(blast_rows)} blasts, {len(few_quakes)} of {len(quake_rows)} earthquakes")
plt.legend()
plt.show()

The earthquakes are spread over the whole box, in broad belts hundreds of kilometres long: those
are the region's active belts. The long one down the coast is the boundary where the Pacific and
North American plates grind past each other — and the first thing this map tells you is that the
boundary is not a line. The scatter filling the right-hand side belongs to it too: the belts
running up the eastern side of the map, inland of the Sierra Nevada, are the same two plates
sliding past each other in the same direction, a few hundred kilometres further east, and a
large share of the motion is taken up out there rather than on the coastal faults. The densest
knot on the whole map sits near 35.8 N, -117.6 — one earthquake sequence, from 2019, that you
meet again later in this notebook — and the tight cluster in the bottom right corner, below the
Salton Sea, is where the boundary itself comes ashore. Only the sparse north-eastern corner of
the box, beyond those belts, is crust that is genuinely pulling apart rather than sliding past
itself, and you can see how few events it holds. The dense knot at about 37.6 N, -118.9 is the
volcanic swarm under Long Valley, and the tight cluster on the coast near 40.3 N, -124.5 is the
Mendocino triple junction, where three plates meet at once. The blasts are not spread at all.
They sit in small tight clumps, because a quarry is a fixed hole in the ground that gets blasted
again and again for decades. That is already a usable clue, and also a warning — a model that
learns *where* the quarries are has learned a list of addresses, not a piece of physics.

Now *how deep*. The two labels are wildly different in number, so `density=True` scales each
histogram to the same total area; without it the blasts would be invisible.

In [ ]:
depth_bins = np.arange(-4, 25.5, 0.5)
plt.hist(quake_rows["depth"], bins=depth_bins, density=True, label="earthquakes")
plt.hist(blast_rows["depth"], bins=depth_bins, density=True, alpha=0.6, label="blasts")
plt.axvline(0, color="black", lw=1)
plt.xlabel("depth (km; negative means above sea level)")
plt.ylabel("share of events per km of depth")
plt.title(f"depth of {len(quake_rows)} earthquakes and {len(blast_rows)} blasts")
plt.legend()
plt.show()

print("median depth — blasts:", blast_rows["depth"].median(),
      " earthquakes:", quake_rows["depth"].median())
print("share deeper than the 25 km the axis reaches — blasts:",
      round((blast_rows["depth"] > 25).mean(), 4),
      " earthquakes:", round((quake_rows["depth"] > 25).mean(), 4))

The blasts pile up to the **left** of the black line. Not at zero — below it, at negative depths,
which sounds like nonsense until you remember that depth in this catalogue is measured down from
sea level and a quarry is a hole in a hillside several hundred metres up. The median blast sits at
-0.519 km, that is 519 metres *above*
sea level, against a median earthquake 6.31 km below it. (The axis stops at
25 km, which leaves 1.3% of the earthquakes off the right-hand side
and no blasts at all.)

That looks like a gift. Look at the actual values before you accept it.

In [ ]:
print(blast_rows["depth"].value_counts().head(5))
print("distinct depth values, blasts:     ", blast_rows["depth"].nunique())
print("distinct depth values, earthquakes:", quake_rows["depth"].nunique())

# How far apart are the blasts that share one repeated depth? Ask in BOTH directions, and in
# kilometres rather than degrees: a degree of latitude is 111.32 km anywhere on Earth, but a
# degree of longitude is that times the cosine of the latitude you are standing at — about 0.8
# of it here — so the two spans cannot be compared until they are converted.
for repeated in blast_rows["depth"].value_counts().index[:2]:
    same_depth = blast_rows[blast_rows["depth"] == repeated]
    lat_span = same_depth["latitude"].max() - same_depth["latitude"].min()
    lon_span = same_depth["longitude"].max() - same_depth["longitude"].min()
    lon_km = lon_span * 111.32 * np.cos(np.deg2rad(same_depth["latitude"].mean()))
    print(repeated, "km:", len(same_depth), "blasts spanning",
          round(lat_span, 2), "deg latitude =", int(round(lat_span * 111.32)), "km, and",
          round(lon_span, 2), "deg longitude =", int(round(lon_km)), "km")

In [ ]:
top_depth = blast_rows["depth"].value_counts().index[0]
second_depth = blast_rows["depth"].value_counts().index[1]

# `place` and `depthError` arrived with the blasts and were dropped when we cut down to COLUMNS.
# `place` reads like "5km NNW of Boron, CA" — but the distance, the bearing, the space after the
# number and even the state's name change from row to row, so counting that text as it stands
# scatters one quarry over dozens of rows. Keep only what sits between " of " and the comma and
# each site is counted once.
blasts_all["site"] = blasts_all["place"].str.split(" of ").str[-1].str.split(",").str[0]

print("the", (blasts_all["depth"] == top_depth).sum(), "blasts at", top_depth, "km:")
print(blasts_all[blasts_all["depth"] == top_depth]["site"].value_counts())
print("the", (blasts_all["depth"] == second_depth).sum(), "blasts at", second_depth, "km:")
print(blasts_all[blasts_all["depth"] == second_depth]["site"].value_counts().head(3))
print(blasts_all["depthError"].value_counts().head(3))

360 separate blasts share the depth -0.82 km — the same value to the
nearest 10 metres, which is as fine as this catalogue quotes a blast depth at all
(95% of them sit on that 10 m grid). The obvious reading is that
they are one quarry, and asking in both directions already strains it. Those 360
events sit within 8 km of each other north to south — the district runs
east–west, so the latitude span was always going to look tight — but
61 km apart east to west, and `place`, one of the columns the archive
sent and we dropped, names two towns: 322 of the blasts at Boron and
38 at Mojave. Two quarries, one number. You could still argue that
away, though. Two pits 61 km apart on the same desert plateau might
genuinely sit at the same height to the nearest 10 metres; that would be a coincidence, not an
impossibility.

The next row down cannot be argued away. 247 blasts share
-0.79 km, and 229 of them —
93% — are at Boron: the same quarry that has
already been handed -0.82 km. One site, two different depths, so neither of them can
be its elevation. The rest of that row is scattered over 637 km of
latitude, from Boron in the southern desert to Shasta Lake in the far
north of the state. And `depthError` settles it. On 2,684 of the
2,803 blasts — 96% — it is the single constant
31.61 km. An uncertainty of 31.61 km attached to an event placed
820 metres above sea level is not a statement about that event; it
is what a fixed depth looks like when the routine that would have solved for one never ran.

**The depth was not measured. It was set** — once an analyst recognises an event as a quarry
blast, they hold its depth at a value they choose. The column is not a property of the ground
shaking at all. It is a note about a decision the analyst had already made.

That is what **leakage** looks like, and it comes with a plain-language test.

> You got 99 percent. Be suspicious. Did one of your columns already know the answer?

Keep depth for now. You will take it away in the homework and see what is left. Last, the clock.

In [ ]:
hour_bins = np.arange(0, 25, 1)
plt.hist(quake_rows["hour"], bins=hour_bins, density=True, label="earthquakes")
plt.hist(blast_rows["hour"], bins=hour_bins, density=True, alpha=0.6, label="blasts")
plt.xlabel("hour of the day, local California time")
plt.ylabel("share of events per hour")
plt.title(f"time of day, {len(quake_rows)} earthquakes and {len(blast_rows)} blasts")
plt.legend()
plt.show()

work = (blast_rows["hour"] >= 10) & (blast_rows["hour"] < 17)
print("between 10 and 17 — blasts:", round(work.mean(), 3), " earthquakes:",
      round(((quake_rows["hour"] >= 10) & (quake_rows["hour"] < 17)).mean(), 3))
print("Monday to Friday  — blasts:", round((blast_rows["weekday"] <= 4).mean(), 3),
      " earthquakes:", round((quake_rows["weekday"] <= 4).mean(), 3))

The earthquakes are flat across the day, which is what a process that has never heard of a clock
should look like. The blasts are a working day: 94.8% of them fall
between 10 in the morning and 5 in the afternoon against 29.4% of the
earthquakes, and 93.5% land Monday to Friday against
70.8%.

## Can a fitted model beat two conditions you wrote by hand?

So the catalogue offers three kinds of clue: an address that repeats, a depth somebody typed in,
and a human timetable. Before any model, the rule you already have in your head:

> Write the dumbest rule you can, first. Any model that cannot beat it is decoration.

That is the **baseline**. Its job is not to be good; its job is to be the number everything
clever has to clear afterwards, because a model that scores below it has taught you nothing
except that you can call `.fit`.

Comparing anything fairly needs data that neither the rule nor the models were allowed to see, so
split the catalogue in two. One argument here is new: `stratify=y` forces the rare class into both
halves in the same proportion. Without it a random 30% could easily take a lopsided share of the
blasts, and the test score would be measuring the split rather than the classifier.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
events = pd.concat([quakes, blasts], ignore_index=True)
events["is_blast"] = events["type"] == "quarry blast"
local = pd.to_datetime(events["time"]).dt.tz_convert("US/Pacific")
events["hour"] = local.dt.hour
events["weekday"] = local.dt.dayofweek
blast_rows = events[events["is_blast"]]
quake_rows = events[events["type"] == "earthquake"]
few_quakes = quake_rows.iloc[::40]

In [ ]:
features = ['latitude', 'longitude', 'depth', 'mag', 'hour', 'weekday']
X = events[features]
y = events["is_blast"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,
                                                    stratify=y)

print("train:", len(X_train), "events,", y_train.sum(), "of them blasts")
print("test: ", len(X_test), "events,", y_test.sum(), "of them blasts")

The two clues that need no computer are the two you just plotted: a blast is at or above sea
level, and a blast happens in the middle of a working day. Write that down as one line of Python.

`&` is the array version of `and`. It asks the question of every row at once, the way `earth < 0`
asked one question of every cell of an elevation grid, and each condition needs its own brackets
because `&` binds more tightly than `<=` does.

### ✏️ Your turn 3

Build the baseline and score it on the held-out half.

`hand_rule` should be `True` where **both** of these hold for a row of `X_test`: `depth` is at or
below 0, and `hour` is at least 10 and less than 17. Then print its precision, its recall and its
F1 against `y_test` — the same three calls as your turn 2.

**Use these names**, because the self-check looks for them: `hand_rule`.

In [ ]:
# ← your answer here


assert len(hand_rule) == len(y_test), "score the rule on X_test, not on the whole catalogue"
print("✓ the baseline — two conditions, F1",
      round(f1_score(y_test, hand_rule), 4), "on", len(y_test), "held-out events")

Two conditions, nothing fitted to anything, F1 0.7718. Write that number down;
everything from here is measured against it. Beating it will have to mean clearing it by more
than the number wanders on its own when the split is re-cut — hold that thought, because you
measure how far it wanders before the end of the hour.

**Logistic regression** is the straight-line fit from earlier in the course, bent to answer a
yes-or-no question.

> The same straight line — but now it outputs a probability between 0 and 1.

Where that line lands is the **decision boundary**: the model answers "blast" on one side of it
and "earthquake" on the other, and a straight boundary is the only thing this model can ever draw.
`.fit(X_train, y_train)` finds it and `.predict` applies it — the same two calls you used to fit a
regression line.

### Predict before you run

Your two hand-written conditions used `depth` and `hour` and scored F1 0.7718. Give
logistic regression exactly the same two columns, and 92,076 labelled events to fit on.
What F1 does it get? Change `my_guess` and run.

In [ ]:
my_guess = None    # ← your number, written down before you look

In [ ]:
assert my_guess is not None, \
    "write a number into my_guess in the cell above — the commitment is the point, "\
    "and a guess you made before you saw the answer is the only one that can teach you anything"
print("✓ committed — I think", my_guess, "is the F1 logistic regression gets on hour and depth")

In [ ]:
model_two_columns = LogisticRegression(max_iter=1000).fit(X_train[["hour", "depth"]], y_train)
guess_two_columns = model_two_columns.predict(X_test[["hour", "depth"]])

print("you guessed:", my_guess)
print("it scored:  ", round(f1_score(y_test, guess_two_columns), 4))
print("it called", guess_two_columns.sum(), "events blasts;", y_test.sum(), "really are")

Not a little worse than two `if` conditions — several times worse. It flagged
278 events as blasts, out of 841 real ones in the held-out half.
Draw the line it found and the reason is visible.

`model_two_columns.coef_[0]` holds one number per column and `model_two_columns.intercept_[0]` the
offset, and the boundary is where they add to zero:
`hour_coef * hour + depth_coef * depth + intercept == 0`. Rearranged for depth, that is a line you
can plot.

In [ ]:
plt.scatter(few_quakes["hour"], few_quakes["depth"], s=3, color="0.7", label="earthquakes")
plt.scatter(blast_rows["hour"], blast_rows["depth"], s=3, color="firebrick", label="blasts")

hours = np.arange(0, 24)
hour_coef, depth_coef = model_two_columns.coef_[0]
boundary = -(hour_coef * hours + model_two_columns.intercept_[0]) / depth_coef
plt.plot(hours, boundary, color="black", lw=2, label="decision boundary")

plt.ylim(-4, 25)
plt.xlabel("hour of the day, local California time")
plt.ylabel("depth (km; negative means above sea level)")
plt.title(f"{len(blast_rows)} blasts and {len(few_quakes)} earthquakes, with the boundary")
plt.legend()
plt.show()

print("at midday the boundary sits at", round(boundary[12], 2), "km")

The model answers *blast* below that line and *earthquake* above it, and the line is almost flat.
It sits at about -0.94 km and barely tilts, which means the model threw the
clock away and kept only *very shallow* — and it put its depth cut well below zero rather than at
zero, because with one blast for every 46 earthquakes it has to be extremely
sure before it dares say blast at all. Almost every red point sits above the line, which is why so
few of them were flagged.

It threw the clock away because it could not use it. Blasts happen in the *middle* of the day, and
a middle is not a side of a line. A straight boundary can say *later than 10* or *earlier than 5*;
saying both at once needs a corner, and a straight line has none. Your two `if` conditions had a
corner. That is the whole difference, and it is what the week on trees and forests comes back for.

Meanwhile the model has been working with one hand tied: it saw two of the six columns you
prepared. Give it all of them.

In [ ]:
model_logistic = LogisticRegression(max_iter=1000).fit(X_train, y_train)
guess_logistic = model_logistic.predict(X_test)

print("accuracy: ", round(accuracy_score(y_test, guess_logistic), 4))
print("precision:", round(precision_score(y_test, guess_logistic), 4))
print("recall:   ", round(recall_score(y_test, guess_logistic), 4))
print("F1:       ", round(f1_score(y_test, guess_logistic), 4))

Hold your reaction to that F1 until you have seen where the mistakes are, because one number hides
which of the two kinds a classifier is making. The **confusion matrix** is the table that
separates them: one row per true label, one column per predicted label, so the off-diagonal cells
are the misses and the false alarms, counted apart.

### ✏️ Your turn 4

Print the confusion matrix of the six-column model with
`confusion_matrix(y_test, guess_logistic)`. It comes back as a 2 by 2 grid of counts: the top row
is the events that really were earthquakes and the bottom row the ones that really were blasts,
and inside each row the first column is "the model said earthquake" and the second "the model said
blast".

Then print the two mistakes on their own lines — the blasts it missed, and the earthquakes it
falsely flagged. Then one more printed line, in words: which of the two mistakes is it making more
of — and which of the two would matter more to somebody verifying a test ban?

**Use these names**, because the self-check looks for them: `matrix`.

In [ ]:
# ← your answer here


assert matrix.sum() == len(y_test), "the matrix should count every held-out event exactly once"
print("✓ the confusion matrix —", matrix[1][1], "blasts caught,", matrix[1][0],
      "missed and", matrix[0][1], "earthquakes falsely flagged")

There is a second way to use the same six columns, and it draws nothing at all.

> What does a quarry blast usually look like? Shallow, weekday, mid-afternoon. Score each clue
> and multiply. Pretending the clues are independent is obviously wrong, and it works anyway.

That is **Naive Bayes**. It learns, one column at a time, what values blasts tend to have and what
values earthquakes tend to have; then for a new event it multiplies the clues together and takes
whichever label comes out ahead. The naive part is the multiplying, which assumes the clues are
independent of one another — and here they plainly are not, since a shallow event in this
catalogue is *more* likely to be at 2pm, not equally likely. `GaussianNB` is the version that
treats each column as a bell curve, and it is used through the identical `.fit` and `.predict`.

### ✏️ Your turn 5

Fit `GaussianNB()` on `X_train` and `y_train`, predict on `X_test`, and print its precision,
recall and F1.

Then print, one per line, the four F1 scores this notebook has produced, so they can be read
together: the always-earthquake rule, your `hand_rule` from your turn 3, the six-column logistic
regression (its predictions are in `guess_logistic`), and this one. Then one line: does either
fitted model clear the hand rule, and by how much?

**Use these names**, because the self-check looks for them: `model_bayes` and `guess_bayes`.

In [ ]:
# ← your answer here


assert len(guess_bayes) == len(y_test), "predict on X_test, the half the model was not fitted to"
print("✓ naive Bayes — F1", round(f1_score(y_test, guess_bayes), 4), "on the same",
      len(y_test), "held-out events")

## Would the same numbers survive a different cut of the data?

One split of one catalogue gives one number, and a number that only holds for the window you
happened to pick is not a result. There are two ways to find out whether these numbers are
results, and the cheaper one first: the hand rule has nothing fitted to anything, so it can be
scored on every event of every year with no risk of cheating. Do that.

You wrote those two conditions once, in your turn 3. From here you need them again on every year,
again on every re-cut split, and twice more in the homework — so give them a name first. `def`
does that, and putting the two hour bounds in as arguments means every line below says out loud
which window it is asking about.

In [ ]:
def two_condition_rule(rows, low, high):
    """True where a row is at or above sea level AND its hour falls inside the window."""
    # the window is an argument rather than fixed at 10 and 17 because you change it later
    return (rows["depth"] <= 0) & (rows["hour"] >= low) & (rows["hour"] < high)


events["year"] = events["time"].str[:4]

for year in range(2015, 2027):
    rows = events[events["year"] == str(year)]
    guess_year = two_condition_rule(rows, 10, 17)
    print(year, len(rows), "events ", rows["is_blast"].sum(), "blasts  Mmax", rows["mag"].max(),
          " precision", round(precision_score(rows["is_blast"], guess_year), 3),
          " recall", round(recall_score(rows["is_blast"], guess_year), 3),
          " F1", round(f1_score(rows["is_blast"], guess_year), 3))

The baseline holds: every year between 0.646 and 0.828, no drift, no
year where it falls apart. Read the last row as what it is, though: the catalogue is cut off at
the end of August, so the 2026 row covers only the first 8
months of that year, and its 113 blasts are not a fall in blasting — a full
year here runs 194 to 310. Its F1 is comparable
with the rest, because a score is a rate and a rate does not care how long you watched. Its
counts are not.

The two lowest F1 scores are 2019 and 2020, and the
same printout says why. Those are the two years the catalogue swells — 23,422 and
20,016 events against 7,168 the year before — and the two years
holding the biggest earthquakes in the file, M7.1 and M6.5. A
large earthquake is followed by aftershocks for months afterwards, and the rule has to say
something about every one of them, while the number of blasts to be caught stays where it was.
Read the two columns and you can see which half of the score gave way: recall barely moves
(0.887 to 0.838 — the same blasts, still caught), while
precision falls from 0.729 to 0.546, because there are
three times as many earthquakes for the rule to be wrong about.

The years were the cheap way to ask. The second way goes at the comparison itself. Your three F1
scores — hand rule, logistic regression, naive Bayes — all came out of one random cut of the
catalogue into two halves, and `random_state=42` is nothing more than the number that decided
which rows went which way. Change it, refit, rescore, and see how much of the difference between
the three was ever there.

Keep the difference itself as you go, in a column of its own. Every split scores the rule and the
model on **the same** held-out events, so subtracting one score from the other on each split is a
fair, like-for-like comparison — which the two columns read separately are not. You have met the
question *is this difference real?* before, and it is a question about this column: how big it
is on average and whether it keeps its sign, not how wide either of the other two columns is.

In [ ]:
def score_one_split(seed):
    """Re-cut the catalogue with a different shuffle and score all three predictors on it."""
    # a new seed sends different rows into each half; stratify keeps the blasts as rare in
    # each half as they are in the whole catalogue, exactly as it did for the pinned split
    X_fit, X_held, y_fit, y_held = train_test_split(X, y, test_size=0.3,
                                                    random_state=seed, stratify=y)
    # the hand rule was fitted to nothing, so it can be scored straight off the held-out half
    rule_f1 = f1_score(y_held, two_condition_rule(X_held, 10, 17))
    # each model is fitted on X_fit alone — then all three are judged on the SAME X_held
    logistic_f1 = f1_score(y_held,
                           LogisticRegression(max_iter=1000).fit(X_fit, y_fit).predict(X_held))
    bayes_f1 = f1_score(y_held, GaussianNB().fit(X_fit, y_fit).predict(X_held))
    return rule_f1, logistic_f1, bayes_f1


gaps = []
for seed in range(10):
    rule_f1, logistic_f1, bayes_f1 = score_one_split(seed)

    gaps.append(rule_f1 - logistic_f1)
    print("split", seed, " hand rule", round(rule_f1, 4), " logistic", round(logistic_f1, 4),
          " naive Bayes", round(bayes_f1, 4),
          " gap", round(rule_f1 - logistic_f1, 4))

print("the gap column — average", round(sum(gaps) / len(gaps), 4),
      " in the rule's favour on", sum(1 for gap in gaps if gap > 0), "of", len(gaps),
      "splits, against it on", sum(1 for gap in gaps if gap < 0))

## The question, answered

**Not from a catalogue.** On the 39,462 held-out events of the pinned split, two
hand-written conditions scored F1 0.7718, logistic regression on six columns scored
0.7701 and naive Bayes 0.7612. Read alone, those three look like an ordering,
and they are not one: the 0.0017 separating rule from model on this split is far
less than either number moves when nothing changes but the shuffle. Across 10 cuts the
hand rule ran 0.7564 to 0.7774 and logistic regression
0.7359 to 0.7744, which between them cover that gap several times
over. Nothing in this notebook says a fitted model beat the baseline.

It does not say the two are level either, and the gap column is why. The rule finished ahead on
8 of the 10 splits, level on 1 and behind on
1, averaging +0.0125. Read the two ranges and you see them
overlap; read the paired difference and you see it keep its sign in 8 draws
out of 10, which is not what a coin flip looks like. So the honest verdict is neither
a victory nor a tie: **two lines of Python are consistently ahead of both fitted models, by an
amount too small to be worth having.** Six columns, 92,076 labelled examples and two
fitted classifiers bought you nothing over two conditions you wrote before any of it — and that,
not a ranking, is the result. None of the three is good enough to hand a decision that matters.

The reason is what the clues are made of. Real discrimination of an explosion from an earthquake
is done on the waveform, not on a catalogue row. An explosion is a sudden push outward from a
point at or near the surface, so it radiates compressional P energy in every direction and makes
comparatively feeble shear and surface waves, while an earthquake is rock sliding past rock on a
fault, which is efficient at making exactly those waves. That contrast is what the ratio of P to S
amplitude and the comparison of body-wave with surface-wave magnitude are built to measure, and
none of it is in the six columns you had. Depth would be a genuine discriminant — nobody buries a
device kilometres down — except that in this file the depth of a blast is not measured but
assigned.

What you classified on instead was a quarry's routine: a fixed address, a working day, a weekday,
and an analyst's convention about depth. Whether any of those four would still be there for the
event you actually care about is the last thing the homework asks you. Either way, this is why
treaty verification reads waveforms rather than catalogues — and why a small F1 on the wrong
features is a more useful thing to report than a large one.

## Week 10 summary

**The question.** Earthquake or explosion — how does the world verify a nuclear test ban?

### What to remember

| | |
|---|---|
| **1** | Accuracy lies when one class is rare: precision and recall are what you report. |
| **2** | If a column already knows the answer, you have leakage rather than a result. |
| **3** | Beat the simplest rule you can write by hand before believing any model. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Logistic regression** | The same straight line — but now it outputs a probability between 0 and 1. |
| **Naive Bayes** | What does a quarry blast usually look like? Shallow, weekday, mid-afternoon. Score each clue and multiply. Pretending the clues are independent is obviously wrong, and it works anyway. |
| **Precision / recall** | Of the ones you flagged, how many were right? Of the real ones, how many did you catch? |
| **Leakage** | You got 99 percent. Be suspicious. Did one of your columns already know the answer? |
| **Baseline** | Write the dumbest rule you can, first. Any model that cannot beat it is decoration. |

### Code you met this week

| Function | What it does |
|---|---|
| `LogisticRegression()` | draw the straight line that best separates two labelled groups |
| `GaussianNB()` | the naive-Bayes classifier — assumes each measurement speaks for itself |
| `accuracy_score(y, pred)` | the fraction it got right — useless when one class is 98% of the data |
| `precision_score(y, pred)` | of the ones it called a blast, how many really were |
| `recall_score(y, pred)` | of the blasts there were, how many it found |
| `f1_score(y, pred)` | precision and recall in one number, so two models can be compared |
| `pd.to_datetime(column)` | turn a column of date text into real dates |
| `column.dt.tz_convert(zone)` | move a timestamp into local time, so 'hour of day' means what it says |
| `train_test_split(X, y, test_size=0.3, random_state=n, stratify=y)` | cut the rows into a training set and a held-out set, keeping the class balance |
| `model.fit(X, y) / model.predict(X)` | learn from the training rows, then label the held-out ones |
| `confusion_matrix(y, pred)` | the two kinds of mistake counted apart: misses in one corner, false alarms in the other |

## Homework

Three parts, all on the table you already have. If you have restarted since class, run the setup
cell at the top, then the checkpoint below, which rebuilds everything class left in memory.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
events = pd.concat([quakes, blasts], ignore_index=True)
events["is_blast"] = events["type"] == "quarry blast"
local = pd.to_datetime(events["time"]).dt.tz_convert("US/Pacific")
events["hour"] = local.dt.hour
events["weekday"] = local.dt.dayofweek
features = ['latitude', 'longitude', 'depth', 'mag', 'hour', 'weekday']
X = events[features]
y = events["is_blast"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,
                                                    stratify=y)


def two_condition_rule(rows, low, high):
    """True where a row is at or above sea level AND its hour falls inside the window."""
    return (rows["depth"] <= 0) & (rows["hour"] >= low) & (rows["hour"] < high)

### ✏️ Your turn 6

Class left one column under suspicion. Take it away and find out what the models were standing on.

Build `features_no_depth` — the same list as `features`, without `"depth"` — then fit a fresh
`LogisticRegression(max_iter=1000)` and a fresh `GaussianNB()` on `X_train[features_no_depth]` and
`y_train`, predict on `X_test[features_no_depth]`, and print the F1 of each. Print beside them the
F1 of the always-earthquake rule, so you have something to read them against; that one needs
`zero_division=0` again. With depth, the two models scored 0.7701 and 0.7612.

Finish by printing one more line that answers the question, in words and on your three numbers:
which of the two models lost more when depth went, and is what either of them has left a
classifier at all?

**Use these names**, because the self-check looks for them: `features_no_depth`,
`f1_logistic_no_depth` and `f1_bayes_no_depth`.

In [ ]:
# ← your answer here


assert "depth" not in features_no_depth, "the point of this part is to leave depth out"
print("✓ without depth — logistic regression F1", round(f1_logistic_no_depth, 4),
      "and naive Bayes F1", round(f1_bayes_no_depth, 4))

### ✏️ Your turn 7

Class chose 10:00 to 17:00 for "working hours" and never argued about it. It is a choice, and this
one is yours. Take **one** of these, not both:

- **wide**, 7 to 19 — catch the early and the late blasts as well
- **narrow**, 11 to 15 — only the solid middle of the day

Set `my_low` and `my_high` to the window you chose, then build `my_rule` by calling the class
function on the held-out half with your two numbers in place of 10 and 17:
`two_condition_rule(X_test, my_low, my_high)`. Print its precision, its recall and its F1. Then
build the 10-to-17 window class used the same way, score it too, and print it underneath so the
two are side by side.

Then say it, in one more printed line: which of precision and recall did your window buy, which
did it sell, and would you defend that trade to somebody who has to act on the flags?

**Use these names**, because the self-check looks for them: `my_low`, `my_high` and `my_rule`.

In [ ]:
# ← your answer here


assert (my_low, my_high) == (7, 19) or (my_low, my_high) == (11, 15), "pick one of the two windows offered — wide is 7 to 19, narrow is 11 to 15 — not the 10 to 17 class used"
print("✓ your window —", my_low, "to", my_high, "gives precision",
      round(precision_score(y_test, my_rule), 4), "and recall",
      round(recall_score(y_test, my_rule), 4))

### ✏️ Your turn 8

Two or three sentences, quoting your own printed numbers.

Your turn 6 gave you an F1 for logistic regression with depth and an F1 for the same model without
it. Quote both, say what the gap between them tells you about what the model had actually learned,
and say whether you would report the first of the two as a result. Then, in one more sentence:
name one clue this week used that would still be there if the event you were trying to catch were
a secret nuclear test rather than a quarry — or say that none would, and why.

*(Double-click this cell and replace this line with your answer.)*